# RAG Self-Consistency
LLM의 확률적 특성을 이용해서, 여러번 답변을 생성하고, 그중에 가장 일관된 답변(다수결)을 채택해서 최종응답으로 사용하는 기법이다.

In [3]:
%pip install sentence_transformers scikit-learn -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [5]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content="파리의 상징은 에펠탑이며, 1889년에 세워졌습니다."),  # 에펠탑 기본 정보
        Document(page_content="파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다."),  # 도시 구조 및 대표 박물관
        Document(page_content="파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.")  # 관광 규모 및 랜드마크
    ]

retrieve_vectordb()

[Document(metadata={}, page_content='파리의 상징은 에펠탑이며, 1889년에 세워졌습니다.'),
 Document(metadata={}, page_content='파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다.'),
 Document(metadata={}, page_content='파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.')]

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser  # 출력 -> 문자열 파싱
from pprint import pprint

llm = init_chat_model('gpt-5.6-luna', n=5)  # 한 번의 요청으로 5개의 응답을 생성
prompt = PromptTemplate.from_template('''
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context로 생성
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

messages = prompt.format_prompt(context=context, question=question).to_messages()
response = llm.generate([messages])
pprint(response)

LLMResult(generations=[[ChatGeneration(text='최종추천일정:\n\n- **1일차:** 오전에는 루브르 박물관에서 파리의 역사와 예술을 감상하고 오후에는 세느강변을 산책하세요.\n- **2일차:** 오전에는 1889년에 세워진 에펠탑을 방문하고 오후에는 주변 전망과 세느강 풍경을 즐기세요.\n- **3일차:** 오전에는 개선문과 샹젤리제 거리를 둘러보고 오후에는 봄이나 가을의 쾌적한 날씨를 만끽하며 파리 시내를 관광하세요.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정:\n\n- **1일차:** 오전에는 루브르 박물관에서 파리의 역사와 예술을 감상하고 오후에는 세느강변을 산책하세요.\n- **2일차:** 오전에는 1889년에 세워진 에펠탑을 방문하고 오후에는 주변 전망과 세느강 풍경을 즐기세요.\n- **3일차:** 오전에는 개선문과 샹젤리제 거리를 둘러보고 오후에는 봄이나 가을의 쾌적한 날씨를 만끽하며 파리 시내를 관광하세요.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0602d-87ac-7dc1-b7b4-e4490b27d669-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1075, 'total_tokens': 1280, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 400}})), ChatGeneration(text='최종추천일정:  \n

In [9]:
pprint(response.generations[0])

[ChatGeneration(text='최종추천일정:\n\n- **1일차:** 오전에는 루브르 박물관에서 파리의 역사와 예술을 감상하고 오후에는 세느강변을 산책하세요.\n- **2일차:** 오전에는 1889년에 세워진 에펠탑을 방문하고 오후에는 주변 전망과 세느강 풍경을 즐기세요.\n- **3일차:** 오전에는 개선문과 샹젤리제 거리를 둘러보고 오후에는 봄이나 가을의 쾌적한 날씨를 만끽하며 파리 시내를 관광하세요.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정:\n\n- **1일차:** 오전에는 루브르 박물관에서 파리의 역사와 예술을 감상하고 오후에는 세느강변을 산책하세요.\n- **2일차:** 오전에는 1889년에 세워진 에펠탑을 방문하고 오후에는 주변 전망과 세느강 풍경을 즐기세요.\n- **3일차:** 오전에는 개선문과 샹젤리제 거리를 둘러보고 오후에는 봄이나 가을의 쾌적한 날씨를 만끽하며 파리 시내를 관광하세요.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0602d-87ac-7dc1-b7b4-e4490b27d669-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1075, 'total_tokens': 1280, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 400}})),
 ChatGeneration(text='최종추천일정:  \n1일차(봄·가을 추천): 세느강 산책으로

In [11]:
candidates = [gen.text for gen in response.generations[0]]
for i, cand in enumerate(candidates):
    print(f'{i+1}: {cand}')
    print()

1: 최종추천일정:

- **1일차:** 오전에는 루브르 박물관에서 파리의 역사와 예술을 감상하고 오후에는 세느강변을 산책하세요.
- **2일차:** 오전에는 1889년에 세워진 에펠탑을 방문하고 오후에는 주변 전망과 세느강 풍경을 즐기세요.
- **3일차:** 오전에는 개선문과 샹젤리제 거리를 둘러보고 오후에는 봄이나 가을의 쾌적한 날씨를 만끽하며 파리 시내를 관광하세요.

2: 최종추천일정:  
1일차(봄·가을 추천): 세느강 산책으로 파리의 역사와 도시 풍경을 살펴본 뒤 루브르 박물관을 관람합니다.  
2일차: 에펠탑(1889년 건립)을 방문하고 주변에서 파리의 상징적인 전경을 감상합니다.  
3일차: 개선문을 둘러본 후 샹젤리제 거리와 주요 도심 관광지를 산책합니다.

3: 최종추천일정:
- 1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 파리의 역사와 도시 발전을 살펴보고 루브르 박물관을 방문합니다.
- 2일차: 1889년에 세워진 에펠탑을 중심으로 주변을 관광하며 파리의 대표적인 상징을 감상합니다.
- 3일차: 개선문을 방문해 파리의 역사적 분위기를 느끼고 도심의 주요 관광지를 둘러보며 여행을 마무리합니다.

4: 최종추천일정:  
- **1일차:** 세느강을 따라 산책하며 루브르 박물관에서 파리의 역사와 예술을 둘러봅니다.  
- **2일차:** 에펠탑을 방문해 1889년 건립된 파리의 상징을 감상하고 주변 세느강 풍경을 즐깁니다.  
- **3일차:** 개선문을 중심으로 파리의 근현대 도시 경관을 둘러보며, 관광객이 비교적 적은 봄·가을에 방문하는 것을 추천합니다.

5: 최종추천일정:
- 1일차: 루브르 박물관에서 파리의 역사와 예술을 둘러본 뒤 세느강 주변을 산책합니다.
- 2일차: 에펠탑을 방문해 1889년부터 이어진 파리의 상징을 감상하고 주변을 관광합니다.
- 3일차: 개선문을 방문해 파리의 대표 관광지를 둘러보며 일정을 마무리합니다.
- 추천 방문 시기: 날씨가 쾌적하고 관광하기 좋은 봄 또는 가을을 추천합니다.



### n개의 응답을 하나로 추출하기

In [ ]:
from langchain_core.output_parsers import BaseOutputParser  # 출력 결과를 원하는 형식으로 변환용 기본 Parser
from sentence_transformers import SentenceTransformer  # 임베딩 모델
from pydantic import Field  # 클래스 속성 정의/검증용
from sklearn.cluster import KMeans  # 클러스터링 모델
from collections import Counter
import numpy as np

class RobustSelfConsistencyParser(BaseOutputParser):
    n_clusters: int = Field(default=2)  # 후보 그룹갯수
    encoder: object = Field(default=SentenceTransformer('all-MiniLM-L6-v2'))  # 임베딩 모델

    def parse(self, generations: list[str]) -> str:
        # 1. 임베딩
        embeddings = self.encoder.encode(generations)  # 벡터로 변환
        # print(embeddings.shape)  # (후보 답변 갯수, 임베딩 벡터 차원)

        # 2. 클러스터링(KMeans)
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        kmeans.fit(embeddings)  # 클러스터링 실행
        # print(kmeans.labels_)   # 각 답변이 어느 라벨에 속해있는지 확인

        # 3. 다수결 투표
        counts = Counter(kmeans.labels_)
        target_label = max(counts, key=counts.get)  # 가장 많은 후보 답변이 속한 클러스터
        # 선택된 클러스터에 속한 후보 답변 인덱스만 추출
        target_indices = np.where(kmeans.labels_ == target_label)[0]
        # print(target_label)
        # print(target_indices)

        # 4. 대표 답변 선택 (중심점에 가장 가까운 후보)
        target_centroid = kmeans.cluster_centers_[target_label]  # 선택된 클러스터 중심점 벡터
        # 후보 답변들과 중심점 사이의 거리를 계산
        distances = np.linalg.norm(embeddings[target_indices] - target_centroid, axis=1)
        representive_idx = np.argmin(distances)  # 중심점과 가장 가까운 답변 인덱스
        return generations[target_indices[representive_idx]]  # 클러스터 중 가장 대표답변 반환

parser = RobustSelfConsistencyParser()
final_answer = parser.parse(candidates)  # 후보 중 대표 답변 선택
print(f"최종 답변 : {final_answer}")

In [ ]:
# 여행 후보 일정을 여러개 생성 후, 클러스터링 기반 Self-Consistency로 최종 답변 생성하는 함수
def travel_planner(question, verbose=False):
    
    retreived_docs = retrieve_vectordb(question)
    context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

    # 프롬프트 템플릿을 메시지 형태로 변환
    messages = prompt.format_prompt(context=context, question=question).to_messages()
    response = llm.generate([messages])  # n=5로 5개 답변 생성
    candidates = [gen.text for gen in response.generations[0]]  # 후보 텍스트들만 추출

    # 눈으로 후보 확인하고 싶으면 verbose True
    if verbose:
        for i, cand in enumerate(candidates):
            print(f"{i+1} : {cand}")
            print()
    
    parser = RobustSelfConsistencyParser()  # 대표 답변 고르는 파서
    return parser.parse(candidates)         # 최종 대표 답변 반환

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'
response = travel_planner(question, verbose=True)
print(f"최종 답변 : {response}")